In [ ]:
import cv2
import numpy as np
import os
import csv
import matplotlib.pyplot as plt

# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "paste the path"
# Main Output Folder
MAIN_OUTPUT_FOLDER = "paste the path"

# =====================================================
# CREATE OUTPUT FOLDERS
# =====================================================

FRAME_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Processed_Frames"
)

MASK_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Segmentation_Masks"
)

FINAL_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Final_Output"
)

FLOWCHART_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Flowcharts"
)

RESULT_FOLDER = os.path.join(
    MAIN_OUTPUT_FOLDER,
    "Quantitative_ResultTables"
)

# Create folders
os.makedirs(FRAME_FOLDER, exist_ok=True)

os.makedirs(MASK_FOLDER, exist_ok=True)

os.makedirs(FINAL_FOLDER, exist_ok=True)

os.makedirs(FLOWCHART_FOLDER, exist_ok=True)

os.makedirs(RESULT_FOLDER, exist_ok=True)


# =====================================================
# IMAGE ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    enhanced = cv2.equalizeHist(blur)

    return enhanced


# =====================================================
# SEGMENTATION
# =====================================================

def segmentation_pipeline(image):

    # Adaptive Threshold
    mask = cv2.adaptiveThreshold(
        image,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    # Morphological Operations
    kernel = np.ones((3, 3), np.uint8)

    opening = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    closing = cv2.morphologyEx(
        opening,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )

    return closing


# =====================================================
# CONTOUR VISUALIZATION
# =====================================================

def draw_segmentation(frame, mask):

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    total_area = 0

    object_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > 100:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                2
            )

            total_area += area

            object_count += 1

    return output, object_count, total_area


# =====================================================
# FLOWCHART GENERATION
# =====================================================

def create_flowchart():

    plt.figure(figsize=(8, 8))

    steps = [
        "Input Video",
        "Frame Extraction",
        "Image Enhancement",
        "Adaptive Thresholding",
        "Morphological Operations",
        "Segmentation Mask",
        "Contour Detection",
        "Final Output"
    ]

    y = list(range(len(steps), 0, -1))

    for i, step in enumerate(steps):

        plt.text(
            0.5,
            y[i],
            step,
            ha='center',
            va='center',
            fontsize=12,
            bbox=dict(boxstyle="round", facecolor="lightblue")
        )

        if i < len(steps) - 1:

            plt.arrow(
                0.5,
                y[i] - 0.3,
                0,
                -0.5,
                head_width=0.02,
                head_length=0.1
            )

    plt.axis('off')

    plt.title("Overall Segmentation Pipeline Flowchart")

    flowchart_path = os.path.join(
        FLOWCHART_FOLDER,
        "segmentation_flowchart.png"
    )

    plt.savefig(flowchart_path)

    plt.close()


# =====================================================
# MAIN PROCESS
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():

        print("Error opening video")
        return

    frame_count = 0

    results = []

    print("Starting full segmentation pipeline...")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # =================================================
        # IMAGE ENHANCEMENT
        # =================================================

        enhanced = enhance_frame(frame)

        # =================================================
        # SEGMENTATION
        # =================================================

        mask = segmentation_pipeline(enhanced)

        # =================================================
        # FINAL OUTPUT
        # =================================================

        final_output, object_count, total_area = draw_segmentation(
            frame,
            mask
        )

        # =================================================
        # SAVE IMAGES
        # =================================================

        cv2.imwrite(
            os.path.join(
                FRAME_FOLDER,
                f"frame_{frame_count}.jpg"
            ),
            enhanced
        )

        cv2.imwrite(
            os.path.join(
                MASK_FOLDER,
                f"mask_{frame_count}.jpg"
            ),
            mask
        )

        cv2.imwrite(
            os.path.join(
                FINAL_FOLDER,
                f"final_{frame_count}.jpg"
            ),
            final_output
        )

        # =================================================
        # STORE QUANTITATIVE RESULTS
        # =================================================

        results.append([
            frame_count,
            object_count,
            total_area
        ])

        # =================================================
        # DISPLAY
        # =================================================

        cv2.imshow("Enhanced Frame", enhanced)

        cv2.imshow("Segmentation Mask", mask)

        cv2.imshow("Final Output", final_output)

        frame_count += 1

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

    # =====================================================
    # SAVE QUANTITATIVE RESULT TABLE
    # =====================================================

    csv_path = os.path.join(
        RESULT_FOLDER,
        "quantitative_results.csv"
    )

    with open(csv_path, mode='w', newline='') as file:

        writer = csv.writer(file)

        writer.writerow([
            "Frame Number",
            "Detected Objects",
            "Total Segmented Area"
        ])

        writer.writerows(results)

    # =====================================================
    # GENERATE FLOWCHART
    # =====================================================

    create_flowchart()

    # =====================================================
    # RELEASE
    # =====================================================

    cap.release()

    cv2.destroyAllWindows()

    print("\nPipeline Completed")

    print("Frames Processed:", frame_count)

    print("\nOutputs saved inside:")
    print(MAIN_OUTPUT_FOLDER)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()